In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS
# ==============================================================================
file_path = 'TELESAPFormulrioMdic-EXPORTAOBOLSISTAAtua_DATA_LABELS_2025-07-23_1329.csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Erro: Arquivo {file_path} não encontrado.")
    raise

# ==============================================================================
# 2. ENGENHARIA DE VARIÁVEIS (TEXT MINING + CHECKBOXES) COM PROTEÇÃO CONTRA NAN
# ==============================================================================
text_cols = [c for c in df.columns if df[c].dtype == object]
df_text = df[text_cols].fillna('')

# Dicionários de RegEx para buscar as comorbidades
regex_has = r'\b(hipertens[aã]o|has|press[aã]o alta)\b'
regex_dm = r'\b(diabetes|dm2|dm1|dm)\b'
regex_dlp = r'\b(colesterol|dislipidemia|dlp|triglic[eé]rides)\b'

# Busca nos textos livres
cond_has_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_has, case=False, regex=True)).any(axis=1)
cond_dm_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_dm, case=False, regex=True)).any(axis=1)
cond_dlp_text = df_text.apply(lambda x: x.astype(str).str.contains(regex_dlp, case=False, regex=True)).any(axis=1)

# Busca nas colunas estruturadas
cols_cronicas = [c for c in df.columns if 'doenças crônicas' in c.lower() and 'choice=' in c.lower()]

cond_has_str = pd.Series(False, index=df.index)
cond_dm_str = pd.Series(False, index=df.index)
cond_dlp_str = pd.Series(False, index=df.index)

if cols_cronicas:
    for c in cols_cronicas:
        is_checked = df[c].astype(str).str.lower().str.strip().isin(['checked', '1', '1.0', 'sim', 'verdadeiro'])
        if 'hipertensão' in c.lower() or 'has' in c.lower(): cond_has_str = cond_has_str | is_checked
        if 'diabetes' in c.lower() or 'dm' in c.lower(): cond_dm_str = cond_dm_str | is_checked
        if 'colesterol' in c.lower() or 'dislipidemia' in c.lower(): cond_dlp_str = cond_dlp_str | is_checked

# Unifica as descobertas
df['Tem_HAS'] = (cond_has_text | cond_has_str).astype(int)
df['Tem_DM'] = (cond_dm_text | cond_dm_str).astype(int)
df['Tem_DLP'] = (cond_dlp_text | cond_dlp_str).astype(int)

# ==============================================================================
# 3. CONSOLIDAÇÃO POR PACIENTE ÚNICO (RECORD ID)
# ==============================================================================
df_pacientes = df.groupby('Record ID')[['Tem_HAS', 'Tem_DM', 'Tem_DLP']].max().reset_index()
df_pacientes = df_pacientes.fillna(0)
total_pacientes = len(df_pacientes)

# ==============================================================================
# 4. CLASSIFICAÇÃO DO RISCO CARDIOVASCULAR
# ==============================================================================
def classificar_risco(row):
    if row['Tem_DM'] == 1 or (row['Tem_HAS'] == 1 and row['Tem_DLP'] == 1):
        return 'Alto Risco (DM ou HAS+DLP)'
    elif row['Tem_HAS'] == 1 or row['Tem_DLP'] == 1:
        return 'Risco Moderado (HAS ou DLP isolado)'
    else:
        return 'Baixo Risco (Sem fatores mapeados)'

df_pacientes['Estratificacao_RCV'] = df_pacientes.apply(classificar_risco, axis=1)

# ==============================================================================
# 4.5. EXPORTAÇÃO DAS BASES DE DADOS (CSVs PARA O ARTIGO)
# ==============================================================================
ordem_risco = ['Alto Risco (DM ou HAS+DLP)', 'Risco Moderado (HAS ou DLP isolado)', 'Baixo Risco (Sem fatores mapeados)']
contagem_risco = df_pacientes['Estratificacao_RCV'].value_counts().reindex(ordem_risco).fillna(0)

# Base 1: Estratificação de Risco
tabela_estratificacao = pd.DataFrame({
    'Nível de Risco Cardiovascular': contagem_risco.index,
    'Pacientes Únicos (N)': contagem_risco.values.astype(int),
    'Proporção da População (%)': ((contagem_risco.values / total_pacientes) * 100).round(2)
})
tabela_estratificacao.to_csv('base_estratificacao_rcv.csv', index=False, encoding='utf-8-sig')

# Base 2: Perfil do Alto Risco
total_dm = int(df_pacientes['Tem_DM'].sum())
total_combo = int(len(df_pacientes[(df_pacientes['Tem_HAS'] == 1) & (df_pacientes['Tem_DLP'] == 1) & (df_pacientes['Tem_DM'] == 0)]))
total_alto_risco = int(contagem_risco.get('Alto Risco (DM ou HAS+DLP)', 0))

if total_alto_risco > 0:
    tabela_alto_risco = pd.DataFrame({
        'Composição do Alto Risco': ['Diabéticos (Com ou sem outras doenças)', 'Combinação HAS + DLP (Sem Diabetes)'],
        'Pacientes (N)': [total_dm, total_combo],
        'Proporção dentro do Alto Risco (%)': [round((total_dm / total_alto_risco) * 100, 2), round((total_combo / total_alto_risco) * 100, 2)]
    })
    tabela_alto_risco.to_csv('base_perfil_alto_risco_rcv.csv', index=False, encoding='utf-8-sig')

print(f"\n{'='*80}")
print("✅ BASES DE DADOS EXPORTADAS COM SUCESSO:")
print("- 'base_estratificacao_rcv.csv' (Dados do Gráfico de Risco)")
if total_alto_risco > 0:
    print("- 'base_perfil_alto_risco_rcv.csv' (Detalhamento do Alto Risco)")
print(f"{'='*80}")

# ==============================================================================
# 5. VISUALIZAÇÃO: PAINEL DE RISCO CARDIOVASCULAR
# ==============================================================================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

paleta_rcV = ['#c0392b', '#f39c12', '#2ecc71']
ax = sns.barplot(x=contagem_risco.values, y=contagem_risco.index, palette=paleta_rcV)

plt.title(f"Estratificação de Risco Cardiovascular da População\n(Base total analisada: {total_pacientes} pacientes únicos)",
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Número de Pacientes Únicos", fontsize=13, fontweight='bold')
plt.ylabel("")

for p in ax.patches:
    qtd = int(p.get_width())
    perc = (qtd / total_pacientes) * 100
    ax.annotate(f" {qtd} pacientes ({perc:.1f}%)",
                (qtd, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points',
                fontweight='bold', fontsize=12, color='#2c3e50')

sns.despine(left=True, bottom=True)
plt.xlim(0, max(contagem_risco.values) * 1.25)
plt.tight_layout()
plt.show()

# ==============================================================================
# 6. RELATÓRIO CLÍNICO DE CUIDADO NO CONSOLE
# ==============================================================================
print(f"\n{'='*90}")
print("DIAGNÓSTICO POPULACIONAL: RISCO CARDIOVASCULAR (RCV)")
print(f"{'='*90}")

alto_risco = int(contagem_risco.get('Alto Risco (DM ou HAS+DLP)', 0))
pct_alto = (alto_risco / total_pacientes) * 100 if total_pacientes > 0 else 0

moderado = int(contagem_risco.get('Risco Moderado (HAS ou DLP isolado)', 0))
pct_mod = (moderado / total_pacientes) * 100 if total_pacientes > 0 else 0

print(f" -> Alto Risco Cardiovascular: {alto_risco} pacientes ({pct_alto:.1f}%)")
print(f" -> Risco Moderado: {moderado} pacientes ({pct_mod:.1f}%)")

print(f"\n[ PERFIL DO ALTO RISCO ]")
if alto_risco > 0:
    print(f" Dos pacientes em alto risco, temos:")
    print(f"  * {total_dm} pacientes carregam o diagnóstico de Diabetes.")
    print(f"  * {total_combo} pacientes não são diabéticos, mas sofrem de Hipertensão + Dislipidemia simultaneamente.")
else:
    print(" Nenhum paciente classificado como Alto Risco.")
print(f"{'='*90}\n")